In [ ]:
!pip install transformers

Looking in indexes: https://pypi.org/simple, https://us-python.pkg.dev/colab-wheels/public/simple/
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.2/7.2 MB 75.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 236.8/236.8 kB 13.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.8/7.8 MB 88.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 71.0 MB/s eta 0:00:00


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


#ViLT

Quando uso ViLT con le immagini stimolo corrispondenti alle frasi di EventsRev, il modello associa lo stesso valore a tutti i pixel_values (il valore 1.). Quale potrebbe essere il motivo e come posso correggerlo?

In [ ]:
from transformers import ViltProcessor, ViltForMaskedLM, ViltImageProcessor
import numpy as np
from PIL import Image
import re
import torch
import cv2

img_w = 384
img_h = 512

path = "/content/drive/MyDrive/Data/EventsRev_picture_stimuli/01_plaus_cop_arrests_criminal.jpg"
image = Image.open(path)
"""image = cv2.imread(path)
image = cv2.cvtColor(image, cv2.IMREAD_COLOR)
resized_image = cv2.resize(image, (img_h, img_w))"""

text = "The cop is arresting the [MASK]"
processor = ViltProcessor.from_pretrained("dandelin/vilt-b32-mlm")
model = ViltForMaskedLM.from_pretrained("dandelin/vilt-b32-mlm")


In [ ]:
encoding = processor(image, text, return_tensors="pt")

In [ ]:
print(encoding.pixel_values) ###

tensor([[[[1., 1., 1.,  ..., 1., 1., 1.],
          [1., 1., 1.,  ..., 1., 1., 1.],
          [1., 1., 1.,  ..., 1., 1., 1.],
          ...,
          [1., 1., 1.,  ..., 1., 1., 1.],
          [1., 1., 1.,  ..., 1., 1., 1.],
          [1., 1., 1.,  ..., 1., 1., 1.]],

         [[1., 1., 1.,  ..., 1., 1., 1.],
          [1., 1., 1.,  ..., 1., 1., 1.],
          [1., 1., 1.,  ..., 1., 1., 1.],
          ...,
          [1., 1., 1.,  ..., 1., 1., 1.],
          [1., 1., 1.,  ..., 1., 1., 1.],
          [1., 1., 1.,  ..., 1., 1., 1.]],

         [[1., 1., 1.,  ..., 1., 1., 1.],
          [1., 1., 1.,  ..., 1., 1., 1.],
          [1., 1., 1.,  ..., 1., 1., 1.],
          ...,
          [1., 1., 1.,  ..., 1., 1., 1.],
          [1., 1., 1.,  ..., 1., 1., 1.],
          [1., 1., 1.,  ..., 1., 1., 1.]]]])


In [ ]:
pixel_values =encoding.pixel_values

In [ ]:
print(np.shape(pixel_values[0]))

torch.Size([3, 384, 512])


In [ ]:


# forward pass

outputs = model(**encoding)

tl = len(re.findall("\[MASK\]", text))

inferred_token = [text]

# gradually fill in the MASK tokens, one by one

with torch.no_grad():

    for i in range(tl):

        encoded = processor.tokenizer(inferred_token)

        input_ids = torch.tensor(encoded.input_ids)

        encoded = encoded["input_ids"][0][1:-1]

        outputs = model(input_ids=input_ids, pixel_values=encoding.pixel_values)

        mlm_logits = outputs.logits[0]  # shape (1, seq_len, vocab_size)


        # only take into account text features (minus CLS and SEP token)

        mlm_logits = mlm_logits[1 : input_ids.shape[1] - 1, :]

        mlm_values, mlm_ids = mlm_logits.softmax(dim=-1).max(dim=-1)


        # only take into account text

        mlm_values[torch.tensor(encoded) != 103] = 0

        select = mlm_values.argmax().item()
        print(select)

        encoded[select] = mlm_ids[select].item()

        inferred_token = [processor.decode(encoded)]

selected_token = ""

encoded = processor.tokenizer(inferred_token)

output = processor.decode(encoded.input_ids[0], skip_special_tokens=True)

print(output)

5
the cop is arresting the man


#FLAVA

Data una frase e l'immagine corrispondente, li diamo in input al modello FLAVA per predire la parola mascherata sulla base del contesto visuale e testuale.

Nel caso in cui si voglia predire la parola mascherata, il modello richiede che venga fornita in input la posizione della parola mascherata nella frase utilizzando input_ids_masked.

Per la parte visuale, il modello estrae direttamente la rappresentazione dall'immagine.

In [ ]:
from PIL import Image
from transformers import FlavaForPreTraining, AutoProcessor

path = "/content/drive/MyDrive/Data/EventsRev_picture_stimuli/01_plaus_cop_arrests_criminal.jpg"
image = Image.open(path)
model = FlavaForPreTraining.from_pretrained("facebook/flava-full")
processor = AutoProcessor.from_pretrained("facebook/flava-full")
text = ["The cop is arresting the [MASK]"]

encoded_inputs = processor(images=image,
                           text=text,
                           padding=True,
                           max_length=77,
                           return_tensors="pt")

input_ids_masked = torch.tensor(encoded_inputs["input_ids"])
attention_mask = torch.tensor(encoded_inputs["attention_mask"])
pixel_values = torch.tensor(encoded_inputs["pixel_values"])

`text_config_dict` is provided which will be used to initialize `FlavaTextConfig`. The value `text_config["id2label"]` will be overriden.
`multimodal_config_dict` is provided which will be used to initialize `FlavaMultimodalConfig`. The value `multimodal_config["id2label"]` will be overriden.
`image_codebook_config_dict` is provided which will be used to initialize `FlavaImageCodebookConfig`. The value `image_codebook_config["id2label"]` will be overriden.
<ipython-input-32-8e65e2c11a8e>:16: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  input_ids_masked = torch.tensor(encoded_inputs["input_ids"])
<ipython-input-32-8e65e2c11a8e>:17: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  attention_

In [ ]:
print(input_ids_masked)

tensor([[  101,  1996,  8872,  2003, 28427,  1996,   103,   102]])


Come si può notare da questa stampa, anche in questo caso i pixel_values restituiti sono un po' strani. Anche se non hanno tutti lo stesso valore, si ripetono gli stessi valori

In [ ]:
print(pixel_values)

tensor([[[[1.9303, 1.9303, 1.9303,  ..., 1.9303, 1.9303, 1.9303],
          [1.9303, 1.9303, 1.9303,  ..., 1.9303, 1.9303, 1.9303],
          [1.9303, 1.9303, 1.9303,  ..., 1.9303, 1.9303, 1.9303],
          ...,
          [1.9303, 1.9303, 1.9303,  ..., 1.9303, 1.9303, 1.9303],
          [1.9303, 1.9303, 1.9303,  ..., 1.9303, 1.9303, 1.9303],
          [1.9303, 1.9303, 1.9303,  ..., 1.9303, 1.9303, 1.9303]],

         [[2.0749, 2.0749, 2.0749,  ..., 2.0749, 2.0749, 2.0749],
          [2.0749, 2.0749, 2.0749,  ..., 2.0749, 2.0749, 2.0749],
          [2.0749, 2.0749, 2.0749,  ..., 2.0749, 2.0749, 2.0749],
          ...,
          [2.0749, 2.0749, 2.0749,  ..., 2.0749, 2.0749, 2.0749],
          [2.0749, 2.0749, 2.0749,  ..., 2.0749, 2.0749, 2.0749],
          [2.0749, 2.0749, 2.0749,  ..., 2.0749, 2.0749, 2.0749]],

         [[2.1459, 2.1459, 2.1459,  ..., 2.1459, 2.1459, 2.1459],
          [2.1459, 2.1459, 2.1459,  ..., 2.1459, 2.1459, 2.1459],
          [2.1459, 2.1459, 2.1459,  ..., 2

In [ ]:
pixel_values.shape

torch.Size([1, 3, 224, 224])

Quando calcoliamo l'output comunichiamo al modello che non vogliamo che calcoli le diverse funzioni di costo. Nel nostro caso, infatti, vogliamo che ci restituisca i logits della funzione di costo calcolata sui token testuali mascherati utilizzando un input multimodale (mmm_text_logits).

In [ ]:
outputs = model(
                input_ids_masked = input_ids_masked,
                attention_mask=attention_mask,
                pixel_values=pixel_values,
                return_loss = False
                )

logits = outputs.mmm_text_logits

#LXMERT

In [ ]:
!git clone https://github.com/huggingface/transformers

Cloning into 'transformers'...
remote: Enumerating objects: 146128, done.
remote: Counting objects: 100% (2477/2477), done.
remote: Compressing objects: 100% (1623/1623), done.
remote: Total 146128 (delta 1302), reused 1668 (delta 743), pack-reused 143651
Receiving objects: 100% (146128/146128), 151.36 MiB | 22.19 MiB/s, done.
Resolving deltas: 100% (107825/107825), done.


In [ ]:
cd transformers

/content/transformers


In [ ]:
ls

awesome-transformers.md  hubconf.py      README_hd.md       setup.py
CITATION.cff             ISSUES.md       README_ja.md       src/
CODE_OF_CONDUCT.md       LICENSE         README_ko.md       templates/
conftest.py              Makefile        README.md          tests/
CONTRIBUTING.md          model_cards/    README_zh-hans.md  utils/
docker/                  notebooks/      README_zh-hant.md
docs/                    pyproject.toml  scripts/
examples/                README_es.md    setup.cfg


In [ ]:
cd examples/research_projects/lxmert

/content/transformers/examples/research_projects/lxmert


In [ ]:
pip install wget

Looking in indexes: https://pypi.org/simple, https://us-python.pkg.dev/colab-wheels/public/simple/
  Preparing metadata (setup.py) ... done
  Created wheel for wget: filename=wget-3.2-py3-none-any.whl size=9657 sha256=d422ca8a639a59b23d9b0f740817eba39a2e827007a6513a07408bfa46fa50d9
  Stored in directory: /root/.cache/pip/wheels/8b/f1/7f/5c94f0a7a505ca1c81cd1d9208ae2064675d97582078e6c769
Successfully built wget


In [ ]:
from IPython.display import clear_output, Image, display
import PIL.Image
import io
import json
import torch
import numpy as np
from processing_image import Preprocess
from visualizing_image import SingleImageViz
from modeling_frcnn import GeneralizedRCNN
from utils import Config
import utils
import wget
import pickle
import os
import cv2
from copy import deepcopy

In [ ]:
torch.cuda.is_available()

False

Estrazione delle feature visuali con la R-CNN

In [ ]:
frcnn_cfg = Config.from_pretrained("unc-nlp/frcnn-vg-finetuned")
frcnn = GeneralizedRCNN.from_pretrained("unc-nlp/frcnn-vg-finetuned", config=frcnn_cfg)
image_preprocess = Preprocess(frcnn_cfg)

%s not found in cache or force_download set to True, downloading to %s https://s3.amazonaws.com/models.huggingface.co/bert/unc-nlp/frcnn-vg-finetuned/config.yaml /root/.cache/torch/transformers/tmpu6c_y_9v


Downloading:   0%|          | 0.00/2.13k [00:00<?, ?B/s]

loading configuration file cache
%s not found in cache or force_download set to True, downloading to %s https://cdn.huggingface.co/unc-nlp/frcnn-vg-finetuned/pytorch_model.bin /root/.cache/torch/transformers/tmponodektl


Downloading:   0%|          | 0.00/262M [00:00<?, ?B/s]

loading weights file https://cdn.huggingface.co/unc-nlp/frcnn-vg-finetuned/pytorch_model.bin from cache at /root/.cache/torch/transformers/57f6df6abe353be2773f2700159c65615babf39ab5b48114d2b49267672ae10f.77b59256a4cf8343ae0f923246a81489fc8d82f98d082edc2d2037c977c0d9d0
All model checkpoint weights were used when initializing GeneralizedRCNN.

All the weights of GeneralizedRCNN were initialized from the model checkpoint at unc-nlp/frcnn-vg-finetuned.
If your task is similar to the task the model of the checkpoint was trained on, you can already use GeneralizedRCNN for predictions without further training.


In [ ]:
image_path = "/content/drive/MyDrive/Data/EventsRev_picture_stimuli/01_plaus_cop_arrests_criminal.jpg"
images, sizes, scales_yx = image_preprocess(image_path)
output_dict = frcnn(
    images,
    sizes,
    scales_yx=scales_yx,
    padding="max_detections",
    max_detections=frcnn_cfg.max_detections,
    return_tensors="pt",
)

/usr/local/lib/python3.10/dist-packages/torch/functional.py:504: UserWarning: torch.meshgrid: in an upcoming release, it will be required to pass the indexing argument. (Triggered internally at ../aten/src/ATen/native/TensorShape.cpp:3483.)
  return _VF.meshgrid(tensors, **kwargs)  # type: ignore[attr-defined]


In [ ]:
# Very important that the boxes are normalized
normalized_boxes = output_dict.get("normalized_boxes")
features = output_dict.get("roi_features")

In [ ]:
normalized_boxes.shape

torch.Size([1, 36, 4])

In [ ]:
features.shape

torch.Size([1, 36, 2048])

In [ ]:
from transformers.models.lxmert.modeling_lxmert import LxmertForPreTraining
from transformers import LxmertTokenizer, LxmertModel
import torch

tokenizer = LxmertTokenizer.from_pretrained("unc-nlp/lxmert-base-uncased")
bert = LxmertForPreTraining.from_pretrained("unc-nlp/lxmert-base-uncased")

In [ ]:
text_sentence = "The cop is arresting the criminal"

In [ ]:
inputs = tokenizer(text_sentence, return_token_type_ids=True, return_attention_mask=True, add_special_tokens=True, return_tensors="pt")

In [ ]:
visual_feats = features
visual_attention_mask = torch.ones(features.shape[:-1], dtype=torch.long)
visual_pos=normalized_boxes

In [ ]:
visual_feats

tensor([[[0.0000e+00, 6.7171e-03, 1.4408e+00,  ..., 0.0000e+00,
          2.6244e-01, 2.0646e-01],
         [2.5251e-02, 0.0000e+00, 2.9588e-02,  ..., 5.6942e-03,
          2.1882e-01, 8.0481e-01],
         [0.0000e+00, 3.9547e-02, 6.3916e-01,  ..., 5.9293e-01,
          2.4805e+00, 1.0319e+00],
         ...,
         [1.0459e-03, 2.3343e-02, 4.0577e-02,  ..., 2.3247e-02,
          6.9307e-03, 2.5627e+00],
         [0.0000e+00, 5.2370e-02, 0.0000e+00,  ..., 0.0000e+00,
          1.1861e-04, 0.0000e+00],
         [3.0997e-02, 2.5140e-02, 2.3576e-03,  ..., 1.0212e+00,
          7.3068e+00, 3.4413e-01]]])

In [ ]:
visual_pos

tensor([[[0.5159, 0.4421, 0.7082, 0.8048],
         [0.4742, 0.0134, 0.7573, 0.8198],
         [0.5695, 0.1287, 0.6485, 0.1862],
         [0.4363, 0.8171, 0.5035, 0.8750],
         [0.0000, 0.0000, 0.5155, 0.7643],
         [0.5292, 0.1528, 0.6762, 0.6693],
         [0.5679, 0.1415, 0.6542, 0.1999],
         [0.5288, 0.7580, 0.5989, 0.8226],
         [0.0000, 0.1325, 0.3681, 0.9469],
         [0.0000, 0.1436, 0.6166, 0.9205],
         [0.4822, 0.1802, 0.7839, 0.9812],
         [0.0000, 0.4396, 0.6549, 1.0000],
         [0.3530, 0.8198, 0.4229, 0.8802],
         [0.0024, 0.0505, 0.2657, 0.8196],
         [0.6829, 0.0099, 0.9937, 0.8603],
         [0.2892, 0.4531, 0.3449, 0.5381],
         [0.0043, 0.0023, 0.4278, 0.3188],
         [0.0000, 0.0000, 0.7746, 0.6416],
         [0.1710, 0.3981, 0.9354, 1.0000],
         [0.3368, 0.1490, 0.7677, 0.8366],
         [0.0435, 0.2728, 0.7914, 0.9763],
         [0.0000, 0.0000, 0.7557, 0.3484],
         [0.2298, 0.0000, 1.0000, 0.5254],
         [0